In [1]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "The customer was charged twice for the same transaction."

tokens = tokenizer.tokenize(text)


tokens

/Users/van/Documents/Van/Code/laya-from-scratch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['the',
 'customer',
 'was',
 'charged',
 'twice',
 'for',
 'the',
 'same',
 'transaction',
 '.']

In [2]:
text = "The customer requested tokenization."

tokenizer.tokenize(text)

['the', 'customer', 'requested', 'token', '##ization', '.']

In [3]:
encoded = tokenizer(
    text,
    return_tensors="pt",
)

encoded

{'input_ids': tensor([[  101,  1996,  8013,  7303, 19204,  3989,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}

In [4]:
encoded["input_ids"]

tensor([[  101,  1996,  8013,  7303, 19204,  3989,  1012,   102]])

In [5]:
tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])

['[CLS]', 'the', 'customer', 'requested', 'token', '##ization', '.', '[SEP]']

In [6]:
backbone = AutoModel.from_pretrained("bert-base-uncased")
outputs = backbone(**encoded)
outputs.last_hidden_state.shape

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 27440.79it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 8, 768])

In [7]:
text = """
The customer was charged twice for the same transaction.
Which department should handle this?
[MASK] Billing
[MASK] Technical support
[MASK] Sales
"""

encoded = tokenizer(
    text,
    return_tensors="pt",
)
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])

tokens

['[CLS]',
 'the',
 'customer',
 'was',
 'charged',
 'twice',
 'for',
 'the',
 'same',
 'transaction',
 '.',
 'which',
 'department',
 'should',
 'handle',
 'this',
 '?',
 '[MASK]',
 'billing',
 '[MASK]',
 'technical',
 'support',
 '[MASK]',
 'sales',
 '[SEP]']

In [8]:
option_mask = encoded["input_ids"] == tokenizer.mask_token_id
option_mask.sum()

tensor(3)

In [9]:
from laya.pretrained import PretrainedDecisionModel

model = PretrainedDecisionModel(
    model_name="bert-base-uncased",
    decision_hidden_dim=128,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20772.15it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
text = """
The customer was charged twice for the same transaction.
Which department should handle this?
[MASK] Billing
[MASK] Technical support
[MASK] Sales
"""

encoded = tokenizer(
    text,
    return_tensors="pt",
)

option_mask = encoded["input_ids"] == tokenizer.mask_token_id

In [12]:
logits, probabilities = model(
    input_ids=encoded["input_ids"],
    attention_mask=encoded["attention_mask"],
    option_mask=option_mask,
)
logits, probabilities

(tensor([[-0.0030,  0.0039,  0.0187]], grad_fn=<SqueezeBackward1>),
 tensor([[0.3302, 0.3325, 0.3374]], grad_fn=<SoftmaxBackward0>))